In [5]:
import sys
import numpy as np
import pandas as pd
import xgboost as xgb
import sklearn as sklearn
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    confusion_matrix
)

import joblib


In [7]:
print("Python:", sys.version)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("xgboost:", xgb.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

Python: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
numpy: 2.4.1
pandas: 2.3.3
xgboost: 3.1.3
scikit-learn: 1.8.0
joblib: 1.5.3


In [2]:
df_train = pd.read_csv("cell2celltrain.csv")
df_test = pd.read_csv("cell2cellholdout.csv")
print(df_train.shape)
print(df_test.shape)


(51047, 58)
(20000, 58)


In [3]:
# Convert target
df_train['Churn'] = df_train['Churn'].map({'Yes': 1, 'No': 0})

X = df_train.drop(columns=['Churn', 'CustomerID'])
y = df_train['Churn']

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [4]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


In [5]:
# Handle class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='aucpr',
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)


scale_pos_weight: 2.47


In [6]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('xgb', xgb_model)
])

model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('xgb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [7]:
calibrated_model = CalibratedClassifierCV(
    model,
    method="isotonic",
    cv=5
)

calibrated_model.fit(X_train, y_train)


,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2","Pipeline(step...=None, ...))])"
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the co

In [8]:
y_val_proba = calibrated_model.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, y_val_proba)
print("ROC AUC:", roc_auc)


ROC AUC: 0.6801634246318571


In [9]:
val_proba = calibrated_model.predict_proba(X_val)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_val, val_proba)

f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Best F1 threshold: {best_threshold:.3f}")


Best F1 threshold: 0.264


In [10]:
for t in [0.25, 0.3, 0.35, 0.4, 0.45]:
    y_pred = (val_proba >= t).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        y_val, y_pred, average="binary"
    )
    print(f"t={t:.2f} | precision={p:.3f} recall={r:.3f} f1={f1:.3f}")


t=0.25 | precision=0.372 recall=0.767 f1=0.501
t=0.30 | precision=0.408 recall=0.623 f1=0.493
t=0.35 | precision=0.449 recall=0.475 f1=0.462
t=0.40 | precision=0.496 recall=0.340 f1=0.403
t=0.45 | precision=0.540 recall=0.216 f1=0.308


In [11]:
val_pred = (val_proba >= best_threshold).astype(int)

print("\nClassification report:")
print(classification_report(y_val, val_pred))

print("ROC-AUC:", roc_auc_score(y_val, val_proba))
print("PR-AUC:", average_precision_score(y_val, val_proba))

print("\nConfusion matrix:")
print(confusion_matrix(y_val, val_pred))



Classification report:
              precision    recall  f1-score   support

           0       0.83      0.53      0.65      7268
           1       0.39      0.72      0.50      2942

    accuracy                           0.59     10210
   macro avg       0.61      0.63      0.58     10210
weighted avg       0.70      0.59      0.61     10210

ROC-AUC: 0.6801634246318571
PR-AUC: 0.45831921746965276

Confusion matrix:
[[3871 3397]
 [ 815 2127]]


In [12]:
best_threshold = 0.30

y_val_pred_best = (y_val_proba >= best_threshold).astype(int)

print(classification_report(y_val, y_val_pred_best))


              precision    recall  f1-score   support

           0       0.81      0.63      0.71      7268
           1       0.41      0.62      0.49      2942

    accuracy                           0.63     10210
   macro avg       0.61      0.63      0.60     10210
weighted avg       0.69      0.63      0.65     10210



In [13]:
X_test = df_test.drop(columns=['CustomerID'])

test_proba = model.predict_proba(X_test)[:, 1]

test_pred = (test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({
    'CustomerID': df_test['CustomerID'],
    'ChurnProbability': test_proba,
    'ChurnPrediction': test_pred
})

submission.head()


,CustomerID,ChurnProbability,ChurnPrediction
0,3000006,0.216274,0
1,3000018,0.363244,1
2,3000034,0.551962,1
3,3000070,0.273577,0
4,3000074,0.213404,0


In [14]:
joblib.dump(
    {
        "model": calibrated_model,
        "threshold": best_threshold,
        "features": X.columns.tolist()
    },
    "churn_model.pkl"
)

print("Saved calibrated churn model with threshold metadata.")


Saved calibrated churn model with threshold metadata.


In [15]:
artifact = joblib.load("churn_model.pkl")
model = artifact["model"]

proba = model.predict_proba(X_val)[:, 1]


print("Loaded model ROC AUC:", roc_auc_score(y_val, proba))


Loaded model ROC AUC: 0.6801634246318571


In [17]:
filepath = "cell2cellholdout.csv"
def preprocess_new_data(file_path, fitted_model_path='churn_model.pkl'):
    df_new = pd.read_csv(file_path)
    
    X_new = df_new.drop(columns=['CustomerID','Churn'], errors='ignore')
    
    artifact = joblib.load(fitted_model_path)
    model = artifact["model"]
    
    probabilities = model.predict_proba(X_new)[:, 1]
    
    return probabilities, df_new['CustomerID']

# Usage:
probs, ids = preprocess_new_data('cell2cellholdout.csv')

In [18]:
filepath = "cell2cellholdout.csv"

def preprocess_new_data(file_path, fitted_model_path='churn_model.pkl'):
    # 1. Load the raw dataset
    df_new = pd.read_csv(file_path)
    
    # 2. Drop non-predictive columns and the empty Churn column
    # This ensures NaNs in 'Churn' don't interfere with the features
    X_new = df_new.drop(columns=['CustomerID', 'Churn'], errors='ignore')
    
    # 3. Load the model and threshold metadata
    artifact = joblib.load(fitted_model_path)
    model = artifact["model"]
    threshold = artifact.get("threshold", 0.3) # Using your saved threshold (default 0.3)
    
    # 4. Generate Predictions
    # The Pipeline automatically handles imputation/scaling for the features
    probabilities = model.predict_proba(X_new)[:, 1]
    
    # 5. Create a readable Results Table
    results = pd.DataFrame({
        'CustomerID': df_new['CustomerID'],
        'Churn_Probability': np.round(probabilities, 4),
        'Risk_Level': ['High' if p >= threshold else 'Low' for p in probabilities]
    })
    
    # --- PRINT OPTIONS ---
    print(f"✅ Processed {len(df_new)} rows from {file_path}")
    print(f"✅ High Risk Customers Detected: {len(results[results['Risk_Level'] == 'High'])}")
    print("-" * 30)
    print("Top 10 Prediction Results:")
    print(results.head(10)) # This prints the table to the console/notebook
    
    return probabilities, df_new['CustomerID']

# Usage:
probs, ids = preprocess_new_data(filepath)

✅ Processed 20000 rows from cell2cellholdout.csv
✅ High Risk Customers Detected: 8709
------------------------------
Top 10 Prediction Results:
   CustomerID  Churn_Probability Risk_Level
0     3000006             0.1117        Low
1     3000018             0.1747        Low
2     3000034             0.2859        Low
3     3000070             0.1271        Low
4     3000074             0.1263        Low
5     3000086             0.2745        Low
6     3000098             0.3438       High
7     3000110             0.2258        Low
8     3000246             0.3308       High
9     3000254             0.1073        Low
